# NSP10–NSP16 Interface: 3D Structural Visualization
**Project:** panCov-rtc-discovery  
**Complex:** NSP10–NSP16 (2'-O-methyltransferase activation)  
**Primary PDB:** 6W4H (SARS-CoV-2, 1.80 Å)  

## Key findings:
- **3 confirmed salt bridges:** ASP106–LYS93, ASP106–LYS95, ASP102–HIS80
- **NSP16 interface: 11/11 hotspots conserved = 1.000**
- **HIS80 inside Zn1 zinc finger loop** — dual mechanism potential
- **fpocket druggability = 0.546** — highest in project
- **Water bridges at HIS80 and LYS93** — displacement pharmacophores

## Views:
1. Full complex  2. Salt bridges  3. Composite score
4. BSA burial  5. Docking box  6. Structural overlay
7. Ranking table  8. Zn1 finger ★

In [1]:
import nglview as nv
import pandas as pd
import json
import numpy as np
from pathlib import Path

PROJECT = Path.home() / 'projects' / 'rtc-pan-coronavirus'
PDB_DIR = PROJECT / '00-reference' / 'pdb_structures'
AF3_DIR = PROJECT / '01-alphafold3' / 'NSP10-NSP16'
RES_DIR = PROJECT / '02-validation' / 'NSP10-NSP16'

# Paths
p_6w4h = str(PDB_DIR / '6W4H.pdb')
p_6wvn = str(PDB_DIR / '6WVN.pdb')
p_6wkq = str(PDB_DIR / '6WKQ.pdb')
p_af3  = str(AF3_DIR / 'NSP10_NSP16_best_model.pdb')

df_rank = pd.read_csv(RES_DIR / 'composite_ranking_NSP10-NSP16_2.csv')
pocket  = json.load(open(RES_DIR / 'pocket_analysis_2.json'))
bsa_raw = json.load(open(RES_DIR / 'bsa_alascan_NSP10-NSP16_2.json'))['bsa']
box     = pocket['docking_box']

# Residue position constants
NSP10_OFFSET = 4270
NSP10_SHIFT  = 17
NSP16_OFFSET = 6797

def g10(local_list):
    return [p + NSP10_OFFSET - NSP10_SHIFT for p in local_list]
def g16(local_list):
    return [p + NSP16_OFFSET for p in local_list]
def sel_resi(chain, resi_list):
    return ' or '.join(':' + str(chain) + ' and ' + str(r) for r in resi_list)

HOTSPOTS_NSP10 = [40,42,43,44,45,71,76,78,80,93,94,95,96]
HOTSPOTS_NSP16 = [40,41,44,76,83,87,102,104,106,244,247]
PRIMARY_NSP10  = {80,93,95}
PRIMARY_NSP16  = {102,106}
ZN1_COORD      = {74,77,83,90}
ZN1_NEIGHBOURS = {71,74,76,77,78,80,83,90,93}

print('Setup complete.')
print(f'Ranking: {len(df_rank)} residues')
print(f'Box center: ({box["center_x"]}, {box["center_y"]}, {box["center_z"]})')

Setup complete.
Ranking: 24 residues
Box center: (75.618, 10.689, 15.511)


## View 1 — Full Complex: All Conserved Hotspots (6W4H)

In [2]:
v1 = nv.show_file(p_6w4h, ext='pdb')
v1.clear_representations()

# Cartoon backbone
v1.add_representation('cartoon', selection=':A', color='#AED6F1', opacity=0.85)
v1.add_representation('cartoon', selection=':B', color='#A9DFBF', opacity=0.85)

# NSP16 hotspots — blue licorice
nsp16_sel = sel_resi('A', g16(HOTSPOTS_NSP16))
v1.add_representation('licorice', selection=nsp16_sel, color='#2C5F8A', radius=0.3)

# NSP10 hotspots — green licorice
nsp10_sel = sel_resi('B', g10(HOTSPOTS_NSP10))
v1.add_representation('licorice', selection=nsp10_sel, color='#27AE60', radius=0.3)

# Primary salt bridge residues — red ball+stick
prim10_sel = sel_resi('B', g10(list(PRIMARY_NSP10)))
prim16_sel = sel_resi('A', g16(list(PRIMARY_NSP16)))
v1.add_representation('ball+stick', selection=prim10_sel, color='#C0392B', radius=0.4)
v1.add_representation('ball+stick', selection=prim16_sel, color='#C0392B', radius=0.4)

v1.center()
v1

NGLWidget()

## View 2 — Salt Bridges Zoomed
**3 confirmed pairs:**
- ASP106(NSP16) — LYS93(NSP10): 3.90 Å [all 4 structures]
- ASP106(NSP16) — LYS95(NSP10): 3.90 Å [all 4 structures]
- ASP102(NSP16) — HIS80(NSP10): 3.33 Å [6WVN + 6WKQ + AF3]

In [3]:
v2 = nv.show_file(p_6w4h, ext='pdb')
v2.clear_representations()

# Muted cartoon
v2.add_representation('cartoon', selection=':A', color='#D5D8DC', opacity=0.3)
v2.add_representation('cartoon', selection=':B', color='#D5D8DC', opacity=0.3)

# NSP16 salt bridge residues — red
sb16_sel = sel_resi('A', [6899, 6903])
v2.add_representation('ball+stick', selection=sb16_sel, color='#C0392B', radius=0.5)
v2.add_representation('spacefill',  selection=sb16_sel, color='#C0392B', radius=0.3)

# NSP10 salt bridge residues — blue
sb10_sel = sel_resi('B', [4333, 4346, 4348])
v2.add_representation('ball+stick', selection=sb10_sel, color='#2980B9', radius=0.5)
v2.add_representation('spacefill',  selection=sb10_sel, color='#2980B9', radius=0.3)

# Zoom to interface
v2.center(sb16_sel)
v2

NGLWidget()

## View 3 — Hotspots Colored by Composite Drug Target Score

In [4]:
v3 = nv.show_file(p_6w4h, ext='pdb')
v3.clear_representations()
v3.add_representation('cartoon', selection=':A', color='#D5D8DC', opacity=0.3)
v3.add_representation('cartoon', selection=':B', color='#D5D8DC', opacity=0.3)

# Color tiers by composite score
high   = df_rank[df_rank['composite'] >= 0.5]
mid    = df_rank[(df_rank['composite'] >= 0.15) & (df_rank['composite'] < 0.5)]
low    = df_rank[(df_rank['composite'] >= 0.05) & (df_rank['composite'] < 0.15)]

for tier, color in [(high,'#C0392B'),(mid,'#E67E22'),(low,'#2C5F8A')]:
    for _, row in tier.iterrows():
        if row['chain'] == 'NSP10':
            gpos = int(row['position']) + NSP10_OFFSET - NSP10_SHIFT
            ch = 'B'
        else:
            gpos = int(row['position']) + NSP16_OFFSET
            ch = 'A'
        sel = f':{ch} and {gpos}'
        v3.add_representation('ball+stick', selection=sel,
                              color=color, radius=0.4)

v3.center()
v3

NGLWidget()

## View 4 — Hotspots Colored by BSA Burial Depth

In [5]:
v4 = nv.show_file(p_6w4h, ext='pdb')
v4.clear_representations()
v4.add_representation('cartoon', selection=':A', color='#D5D8DC', opacity=0.3)
v4.add_representation('cartoon', selection=':B', color='#D5D8DC', opacity=0.3)

bsa_map = {}
for key, val in bsa_raw.items():
    ch, pos = key.split(',')
    bsa_map[(ch.strip(), int(pos))] = float(val)

max_bsa = max(bsa_map.values()) if bsa_map else 1

for (chain, pos), bsa_val in bsa_map.items():
    norm = bsa_val / max_bsa
    if norm >= 0.6:
        color = '#148F77'
    elif norm >= 0.3:
        color = '#2980B9'
    elif norm >= 0.1:
        color = '#F39C12'
    else:
        continue
    if chain == 'B':
        gpos = pos + NSP10_OFFSET - NSP10_SHIFT
    else:
        gpos = pos + NSP16_OFFSET
    v4.add_representation('ball+stick',
                          selection=f':{chain} and {gpos}',
                          color=color, radius=0.35)

v4.center()
v4

NGLWidget()

## View 5 — Docking Box (primary hotspot-centered)

In [6]:
v5 = nv.show_file(p_6w4h, ext='pdb')
v5.clear_representations()
v5.add_representation('cartoon', selection=':A', color='#AED6F1', opacity=0.7)
v5.add_representation('cartoon', selection=':B', color='#A9DFBF', opacity=0.7)

# Primary residues
v5.add_representation('ball+stick',
    selection=sel_resi('B', [4333, 4346, 4348]), color='#C0392B', radius=0.4)
v5.add_representation('ball+stick',
    selection=sel_resi('A', [6903, 6899]), color='#C0392B', radius=0.4)

# Docking box as shape
cx,cy,cz = box['center_x'],box['center_y'],box['center_z']
sx,sy,sz = box['size_x']/2,box['size_y']/2,box['size_z']/2

shape = v5.shape
corners = [
    [cx-sx,cy-sy,cz-sz],[cx+sx,cy-sy,cz-sz],
    [cx+sx,cy+sy,cz-sz],[cx-sx,cy+sy,cz-sz],
    [cx-sx,cy-sy,cz+sz],[cx+sx,cy-sy,cz+sz],
    [cx+sx,cy+sy,cz+sz],[cx-sx,cy+sy,cz+sz],
]
edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),
         (6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
for i,j in edges:
    shape.add_cylinder(corners[i], corners[j],
                       [0.9,0.5,0.1], 0.3)

v5.center()
v5

NGLWidget()

## View 6 — Structural Overlay: 6W4H + 6WVN + 6WKQ + AF3
Confirms salt bridge conservation across species and prediction methods.

In [7]:
# Structural overlay using show_structure_file (most compatible)
v6 = nv.show_file(p_6w4h, ext='pdb')
v6.clear_representations()

# Add other structures as components
v6.add_component(p_6wvn, ext='pdb')
v6.add_component(p_6wkq, ext='pdb')
v6.add_component(p_af3,  ext='pdb')

# Style each component by index
# Component 0 = 6W4H (blue, primary)
v6._remote_call('setRepresentation',
    target='compList', args=['cartoon'],
    kwargs={'color':'#2C5F8A','opacity':0.85,'component_index':0})

# Salt bridges on 6W4H
sb16 = sel_resi('A', g16([102,106]))
sb10 = sel_resi('B', g10([80,93,95]))
v6.add_representation('ball+stick', selection=sb16,
                      color='#C0392B', radius=0.5, component=0)
v6.add_representation('ball+stick', selection=sb10,
                      color='#2C5F8A', radius=0.5, component=0)

# Style components 1,2,3
for i, color in enumerate(['#C0392B','#27AE60','#F39C12'], 1):
    v6.add_representation('cartoon', color=color,
                          opacity=0.45, component=i)

v6.center()
print('Structural overlay:')
print('  BLUE  = 6W4H SARS-CoV-2 (1.80 Å) primary')
print('  RED   = 6WVN SARS-CoV-1 (1.95 Å)')
print('  GREEN = 6WKQ SARS-CoV-2 (2.05 Å)')
print('  GOLD  = AF3 model (iptm=0.79)')
v6

Structural overlay:
  BLUE  = 6W4H SARS-CoV-2 (1.80 Å) primary
  RED   = 6WVN SARS-CoV-1 (1.95 Å)
  GREEN = 6WKQ SARS-CoV-2 (2.05 Å)
  GOLD  = AF3 model (iptm=0.79)


NGLWidget()

## View 7 — Complete Drug Target Ranking Table

In [8]:
from IPython.display import display

display_cols = ['residue','chain','bsa','contact_score',
                'conservation','total_loss','composite',
                'primary_sb','zn1_region']
df_show = df_rank[display_cols].copy()
df_show.columns = ['Residue','Chain','BSA(Å²)','Contact',
                   'Conservation','AlaLoss','Score',
                   'PrimarySB','Zn1Region']

def highlight(row):
    if row['PrimarySB']:
        return ['background-color:#FADBD8']*len(row)
    elif row['Zn1Region']:
        return ['background-color:#FDEBD0']*len(row)
    elif row['Chain'] == 'NSP16':
        return ['background-color:#D6EAF8']*len(row)
    return ['']*len(row)

styled = (df_show.style
          .apply(highlight, axis=1)
          .format({'Score':'{:.4f}','BSA(Å²)':'{:.1f}',
                   'Conservation':'{:.3f}'})
          .set_caption(
              'NSP10-NSP16 Drug Target Ranking | '
              'Red=primary SB | Orange=Zn1 | Blue=NSP16'))
display(styled)
print(f'Primary SB residues: '
      f'{list(df_rank[df_rank.primary_sb==True].residue)}')
print(f'Zn1 region: '
      f'{list(df_rank[df_rank.zn1_region==True].residue)}')

,Residue,Chain,BSA(Å²),Contact,Conservation,AlaLoss,Score,PrimarySB,Zn1Region
0,LEU45,NSP10,175.5,67.000000,1.000,24,1.0000,False,False
1,LYS93,NSP10,102.7,22.500000,1.000,8,0.2130,True,True
2,MET44,NSP10,61.1,27.000000,1.000,8,0.1811,False,False
3,VAL42,NSP10,73.7,30.000000,0.689,10,0.1551,False,False
4,LYS43,NSP10,113.9,21.000000,1.000,2,0.1400,False,False
5,ASN40,NSP10,94.9,19.000000,1.000,0,0.1092,False,False
6,ALA71,NSP10,37.3,14.500000,1.000,0,0.0787,False,True
7,GLY94,NSP10,69.6,14.500000,1.000,0,0.0756,False,False
8,ARG78,NSP10,74.8,7.500000,1.000,0,0.0479,False,True
9,LYS95,NSP10,44.8,6.500000,1.000,3,0.0342,True,False


Primary SB residues: ['LYS93', 'LYS95', 'HIS80', 'ASN102', 'SER106']
Zn1 region: ['LYS93', 'ALA71', 'ARG78', 'HIS80', 'TYR76', 'ASP76']


## View 8 ★ — Zn1 Zinc Finger: Coordination + Interface Context

**HIS80** (primary salt bridge partner of ASP102) sits **inside the Zn1 finger loop**.

| Color | Meaning |
|-------|---------|
| 🟡 Gold | Zn1 coordinators: CYS74, CYS77, HIS83, CYS90 |
| 🔴 Red | HIS80 — salt bridge anchor inside Zn1 loop |
| 🟣 Purple | ASP102 (NSP16) — salt bridge partner |
| 🔵 Cyan sphere | Zn atom |

**Drug implication:** Targeting HIS80–ASP102 = PPI disruption + potential Zn perturbation

In [9]:
v8 = nv.show_file(p_6w4h, ext='pdb')
v8.clear_representations()

# Full complex — very muted
v8.add_representation('cartoon', selection=':A', color='#D5D8DC', opacity=0.2)
v8.add_representation('cartoon', selection=':B', color='#D5D8DC', opacity=0.2)

# Zn1 neighbourhood cartoon — gold
zn1nb_sel = sel_resi('B', [4324, 4327, 4329, 4330, 4331, 4333, 4336, 4343, 4346])
v8.add_representation('cartoon', selection=zn1nb_sel,
                      color='#F39C12', opacity=0.9)

# Zn1 coordinators — gold ball+stick
zn1_sel = sel_resi('B', [4343, 4336, 4330, 4327])
v8.add_representation('ball+stick', selection=zn1_sel,
                      color='#F39C12', radius=0.4)

# HIS80 — red (salt bridge + Zn1 context)
v8.add_representation('ball+stick', selection=':B and 4333',
                      color='#C0392B', radius=0.55)
v8.add_representation('spacefill',  selection=':B and 4333',
                      color='#C0392B', radius=0.35)

# ASP102 NSP16 — purple
v8.add_representation('ball+stick', selection=':A and 6899',
                      color='#7D3C98', radius=0.55)
v8.add_representation('spacefill',  selection=':A and 6899',
                      color='#7D3C98', radius=0.35)

# Zn atom — cyan spacefill
v8.add_representation('spacefill', selection=':B and ZN',
                      color='#00BCD4', radius=1.2)

# Zoom to Zn1 region
v8.center(zn1nb_sel)
v8

NGLWidget()

## Ranking Table — NSP10-NSP16 [updated]

| Rank | Residue | BSA | Loss | Cons | Score | Flags |
|------|---------|-----|------|------|-------|-------|
| 1 | NSP10-GLU45 | 175.5 | 24 | 1.000 | 1.0000 |  |
| 2 | NSP10-VAL93 | 102.7 | 8 | 1.000 | 0.2130 | ★SB |
| 3 | NSP10-PRO44 | 61.1 | 8 | 1.000 | 0.1811 |  |
| 4 | NSP10-VAL42 | 73.7 | 10 | 0.689 | 0.1551 |  |
| 5 | NSP10-THR43 | 113.9 | 2 | 1.000 | 0.1400 |  |
| 6 | NSP10-ILE40 | 94.9 | 0 | 1.000 | 0.1092 |  |
| 7 | NSP10-PRO71 | 37.3 | 0 | 1.000 | 0.0787 |  |
| 8 | NSP10-GLY94 | 69.6 | 0 | 1.000 | 0.0756 |  |
| 9 | NSP10-LYS78 | 74.8 | 0 | 1.000 | 0.0479 |  |
| 10 | NSP10-PHE95 | 44.8 | 3 | 1.000 | 0.0342 | ★SB |
| 11 | NSP10-THR96 | 61.5 | 4 | 0.172 | 0.0223 |  |
| 12 | NSP10-LYS80 | 103.3 | 0 | 1.000 | 0.0213 | ★SB |
| 13 | NSP10-ASP76 | 32.3 | 0 | 1.000 | 0.0000 |  |
| 14 | NSP10-ILE40 | 94.9 | 0 | 1.000 | 0.0000 |  |
| 15 | NSP16-ILE41 | 67.5 | 8 | 1.000 | 0.0000 |  |
